# Lab 2: Working with Anthropic API and Reasoning Models

### Overview

Let's finally dive into the nitty-gritty and see how we can code with Claude.

This notebook explores the Anthropic API, focusing on how to leverage both standard chat capabilities and advanced reasoning features. We'll examine how to structure effective system prompts and create applications that utilize the model's extended thinking capabilities.

### Objectives
- Set up and configure the Anthropic API
- Create tailored system prompts for specific use cases
- Compare standard responses with reasoning-enhanced outputs
- Analyze the "thinking" process in reasoning models

#### Step 1: Import libraries and load the environment variables

In [5]:
import os
import anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY is None:
    raise Exception("API key is missing")

#### Step 2: Call the API

In [16]:
client = anthropic.Client()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    system="don't answer any questions",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ]
)

print(response.content[0].text)


The capital of France is Paris.


#### Step 3: Customize the Prompt

Customer service representative

In [17]:
complaint = "You have over charged me for this product. I demand a refund."

In [18]:
company_policy = """
- We offer refunds only within 14 days of purchase.
- For purchases over $100, a supervisor must approve the refund.
- We prioritize customer satisfaction but aim to minimize refund abuse.

"""

In [22]:
system_message = "You are a customer service agent."

prompt = f"""
        We have user complaint : "{complaint}"
        company policy is as follows : "{company_policy}"


        Your job is to give a response to the user complaint based on the company policy.
        keep your tone professional and empathetic, but also firm about the company policy.

"""
message = client.messages.create(
    model = "claude-haiku-4-5",
    max_tokens = 300,
    system = system_message,
    messages=[
        {
            "role" : "user",
            "content" : prompt
        }
    ]
)
display(Markdown(message.content[0].text))


# Response to Customer Complaint

Dear Valued Customer,

Thank you for reaching out, and I sincerely apologize that you feel overcharged for your purchase. I understand how frustrating this situation must be, and I'm here to help resolve it.

To assist you further, I'll need to gather some information:

1. **Purchase Date**: When did you make this purchase?
2. **Order Amount**: What was the total purchase price?
3. **Details of the Overcharge**: Could you explain specifically what led you to believe you were overcharged? (e.g., price discrepancy, duplicate charges, unexpected fees)

**Regarding Our Refund Policy:**

Our standard refund window is **14 days from the date of purchase**. If your purchase falls within this timeframe, I can absolutely help process a refund request.

Additionally, for purchases exceeding $100, supervisor approval is required as part of our quality assurance process. Should your situation warrant a refund, I will escalate your case to management to ensure it receives proper attention.

**Next Steps:**

Please reply with the information above at your earliest convenience, and I will:
- Review your case thoroughly
- Verify the charge details
- Work toward a swift resolution

We truly value your business and are committed to making this right within our policy guidelines.

Warm regards,  
**Customer

#### Step 4: Rate the response using a reasoning model

In [28]:
system_message = "You are a senior customer service agent with advanced reasoning capabilities. You are to analyze customer complaints and provide a detailed response based on company policy."

reasoning_prompt = f"""
A customer has made a complaint.

{complaint}

Evaluate this customer service email response.

{message.content[0].text}

Company Policy:
{company_policy}

Based on this, decide how the support team should respond. Your analysis must include:
- Whether the response is appropriate given the complaint and company policy
- Professional tone and empathy

"""

reasoning_message = client.messages.create(
    model="claude-sonnet-5-5",

    thinking={
        "type": "adaptive"
    },

    output_config={
        "effort": "high"
    },

    max_tokens=3000,

    system=system_message,

    messages=[
        {
            "role": "user",
            "content": reasoning_prompt
        }
    ]
)

reasoning_message = reasoning_message.content
print(type(reasoning_message))

for block in reasoning_message:
    if block.type == "text":
        display(Markdown(block.text))


<class 'list'>


# Evaluation

## Overall verdict
**Mostly appropriate, but it needs revision before sending.** The response follows company policy and avoids promising a refund prematurely. It has a few flaws in completeness, tone, and professionalism.

---

## 1. Policy alignment

| Policy element | Handled? | Notes |
|---|---|---|
| Refunds only within 14 days | ✅ Yes | Stated accurately and conditionally. |
| Supervisor approval over $100 | ✅ Yes | Stated accurately. Calling it a "quality assurance process" is embellishment not found in policy. |
| Prioritize satisfaction, minimize abuse | ✅ Mostly | Asking for purchase details and verifying charges is a sound, non-accusatory abuse safeguard. |

**Strengths**
- It doesn't approve or deny the refund before the facts are known, which is correct because the complaint has no date, amount, or order details.
- It collects exactly what's needed to apply the two policy rules: purchase date (14-day window) and amount (the $100 threshold).
- It sets expectations without overpromising.

**Weaknesses**
- **It conflates an overcharge with a standard refund.** The customer alleges a billing error, not a change of mind. The response leads with the 14-day window, which could read as "you're out of luck if you're past 14 days." Policy doesn't say how billing errors are treated, so the agent shouldn't imply a denial. The response should frame this as a charge review first, and the refund-window question should be flagged internally if the purchase turns out to be older than 14 days.
- **It doesn't request an order number, receipt, or account email.** Without one of these, the agent can't verify the charge.
- **It gives no follow-up timeframe** (e.g., "within 1–2 business days of your reply").

---

## 2. Tone and empathy

**Strengths**
- It opens with an apology and acknowledges the frustration.
- The language is polite, calm, and solution-oriented ("make this right," "swift resolution").
- It doesn't get defensive or accuse the customer of abuse.

**Weaknesses**
- **"I apologize that you feel overcharged"** is a hedged apology. It subtly questions the customer's claim. "I'm sorry for the concern about your charge" or "I'm sorry you were charged incorrectly, and I'll look into it" is better.
- **"Dear Valued Customer"** and "Customer" in the sign-off feel generic and templated.
- **The signature is broken.** It ends with `**Customer` with unclosed formatting, and there's no agent name, title, or contact information. This is the most visible professionalism error. It looks unfinished and makes the sender appear to be "Customer."
- The policy section is somewhat heavy for a first reply to an upset customer. It should be shorter and placed after the empathy and information request.

---

## 3. Recommended approach for the support team

1. **Acknowledge and apologize sincerely** without conceding or disputing the claim.
2. **Request identifying details:** order number or account email, purchase date, amount charged, expected amount, and a receipt or statement screenshot if available.
3. **Verify internally** against billing records.
   - **Confirmed billing error:** correct it. Escalate to a supervisor if the amount exceeds $100, and flag for guidance if the purchase is outside 14 days, since policy doesn't address that case.
   - **Standard refund request within 14 days:** process it, with supervisor approval if over $100.
   - **Outside 14 days and no billing error:** politely decline, explain the policy, and offer alternatives where available.
4. **Give a response timeline.**

---

## 4. Suggested revised response

> **Subject: Your billing concern – let's get this sorted out**
>
> Dear [Customer Name],
>
> Thank you for contacting us, and I'm sorry for the concern about your charge. I understand how frustrating it is to see an amount you didn't expect, and I'd like to look into it right away.
>
> To review your account, could you please reply with:
> 1. Your **order number** (or the email address used for the purchase)
> 2. The **purchase date**
> 3. The **amount you were charged** and the amount you expected to pay
> 4. A brief description of the issue (e.g., price difference, duplicate charge, unexpected fee). A receipt or statement screenshot is helpful if you have one.
>
> Once I have this, I'll verify the charge against our records. For reference, our refund policy covers requests made within 14 days of purchase, and refunds on purchases over $100 require supervisor approval. If that applies here, I'll handle the escalation so you don't need to do anything extra.
>
> I'll follow up within [1–2 business days] of receiving your details.
>
> Thank you for your patience. We want to get this right.
>
> Kind regards,
> [Agent Name]
> Customer Support Team | [Contact info]

---

## Summary

| Criterion | Rating |
|---|---|
| Policy accuracy | Good |
| Appropriateness (no premature approval or denial) | Good |
| Completeness (order ID, timeline, billing-error distinction) | Needs improvement |
| Empathy | Adequate, with a slightly hedged apology |
| Professionalism | Needs improvement (generic greeting, broken signature) |

In [ ]:
# reasoning_message = reasoning_message[1].text

 

# **Your Challenge!!!**

A customer has submitted a complaint. Based on company policy and the situation, the model must select the correct resolution option from a list, and give a 1-line justification.

In [30]:
complaint = "I ordered a ceramic mug and received a broken one. Please fix this!"

policy = """
- Broken or damaged items must be reported within 7 days of delivery.
- If reported in time, the item is replaced or refunded.
- No action is taken for reports after 7 days unless escalated.
"""

metadata = """
    Reported Days After Delivery: 5
    Item Value: $12
    Customer Tier: Standard
    Past Complaints: 0
"""


Build a customer support agent that could help solve this issue

The agent should choose one of the following options:
- Refund Item
- Replace Item
- Offer Discount
- Apologize (no action)
- Report to supervisor

In [37]:
system_message = "You are a senior in the customer satisifaction team"

prompt = F"""
customer has made a complaint {complaint}

company policy as follows : {policy}

here is the actual reported day and discription of the product : {metadata}

You are a customer side agent , decide how the support team should respond. Your analysis must include:
- Whether the response is appropriate given the complaint and company policy
- Professional tone and empathy 
"""


message = client.messages.create(
    model = 'claude-sonnet-5-5',
    thinking = {
        "type": "adaptive"
    },
     output_config={
            "effort": "high"
    },
    system = system_message,
    max_tokens = 3000,
    messages = [
        {
            "role" : "user",
            "content" : prompt
        }
    ]
)
reasoning_message = message.content
for block in reasoning_message :
    if block.type == 'text' :
        display(Markdown(block.text))

# Case Decision: Broken Ceramic Mug

## 1. Policy Assessment

| Factor | Finding | Impact |
|---|---|---|
| Reported timing | Day 5 (limit is 7 days) | **Within the window, so eligible** |
| Issue type | Broken item on arrival | Covered by policy |
| Item value | $12 | Low financial risk |
| Customer tier / history | Standard, 0 prior complaints | No abuse or fraud indicators |
| Escalation needed? | No | Policy applies directly |

**Decision: Approve a replacement or full refund, at the customer's choice.** Nothing in the policy gives grounds to deny, delay, or add hurdles.

### Handling notes
- **Don't require a return of the broken item.** Return shipping would cost more than the mug is worth and would add frustration.
- **A photo is optional.** It's useful for internal damage tracking and carrier claims, but resolution should not depend on it. For a $12 item with a clean history, requiring proof risks making a loyal customer feel accused.
- **Offer a choice.** Let the customer pick a replacement or a refund. Process it promptly.

## 2. Recommended Response to the Customer

> **Subject: Your broken mug – we'll make this right**
>
> Hi [Customer Name],
>
> I'm so sorry your ceramic mug arrived broken. That's frustrating, especially when you were looking forward to using it, and I appreciate you letting us know.
>
> Since you reported this within our 7-day window, I'm happy to help right away. You can choose either:
>
> 1. **A free replacement mug**, shipped at no cost to you, or
> 2. **A full refund** of $12 to your original payment method.
>
> There's no need to send the damaged mug back. Just reply with your preferred option and we'll process it immediately. If you have a photo of the damage and packaging handy, feel free to attach it. It helps us improve our packaging, but it's not required.
>
> Thank you for your patience, and again, I'm sorry for the trouble.
>
> Warm regards,
> [Agent Name]
> Customer Satisfaction Team

## 3. Why This Response Works

- **Policy-compliant:** It applies the replace-or-refund remedy for a claim made within 7 days.
- **Empathetic:** It opens with a sincere apology and acknowledges the customer's disappointment without being excessive.
- **Low-effort for the customer:** It gives clear options, requires no return, and needs one reply to resolve.
- **Professional:** It is concise, specific, and free of defensive language or policy jargon.
- **Proportionate:** The remedy fits the $12 value and the customer's clean history, and preserves goodwill.

## 4. Internal Follow-Up
- Log the damage claim against the order and carrier.
- Flag the packaging if ceramic breakage claims recur.
- If the customer chooses a replacement, make sure it ships with extra protective packaging.

## **Reflections**

1. What reasoning steps did the model take to select its resolution? Were they logical and aligned with company policy? (Hint: Have a look at the reasoning output)
2. If you were designing the agent, what would you change about its decision-making process?
3. What did you learn about prompt design and reasoning models from this exercise?